# Benchmark Qwen2-VL-2B — 355 keyframe AIC

Chay lai benchmark tren toan bo anh de vuot nguong 100 anh ma de bai yeu cau
(hien `sample_results.json` moi co 79 anh cho model nay).

⚠️ **355 anh nay gom ca 290 anh da dung train QLoRA.** Voi bai nop thi vo hai —
model goc chua nap LoRA. Nhung KHONG dung tap nay de do truoc/sau khi nap LoRA:
phai dung 60 anh holdout rieng.


In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Chua bat GPU'


In [ ]:
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes qwen-vl-utils
print('Cai xong')
import transformers
print('transformers:', transformers.__version__)


In [ ]:
import subprocess, sys
from pathlib import Path

# Clone tu fork public -- code luon khop ban moi nhat da day len.
REPO = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
NHANH = 'research/vlm-prompting'
DICH = Path('/kaggle/working/repo')

if not DICH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', NHANH, REPO, str(DICH)],
                   check=True)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'
assert PKG.exists(), f'Khong thay code tai {PKG}'
sys.path.insert(0, str(PKG))
print('Code tai:', PKG)

hash_code = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           cwd=DICH, capture_output=True, text=True).stdout.strip()
print('Commit:', hash_code)


In [ ]:
ANH_DIR = next(Path('/kaggle/input').glob('**/images'), None)
print('Thu muc anh:', ANH_DIR)
so_anh = len(list(ANH_DIR.glob('*.jpg')))
print('So anh:', so_anh)
assert so_anh >= 100, f'Chi co {so_anh} anh, de bai can >= 100'


In [ ]:
# --strict: model khong tai duoc thi NEM LOI, khong am tham roi ve mock.
# Goi qua subprocess (khong dung !shell) de duong dan Python noi suy dung.
lenh = [
    sys.executable, 'scripts/benchmark_runner.py',
    '--mode', 'mass',
    '--models', 'qwen2vl-2b',
    '--backend', 'transformers',
    '--strict', '--restart',
    '--frames-dir', str(ANH_DIR),
    '--out-dir', '/kaggle/working/ket_qua',
]
print('Chay:', ' '.join(lenh))
kq = subprocess.run(lenh, cwd=str(PKG))
assert kq.returncode == 0, f'benchmark_runner loi, ma thoat {kq.returncode}'


In [ ]:
import json

ra = Path('/kaggle/working/ket_qua')
print('File sinh ra:')
for f in sorted(ra.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(ra)}  {f.stat().st_size:,} bytes')

kq_file = ra / 'sample_results.json'
assert kq_file.exists(), 'Khong thay sample_results.json'

d = json.loads(kq_file.read_text(encoding='utf-8'))
muc = d if isinstance(d, list) else d.get('results', d)
anh = {m['image'] for m in muc}
print()
print(f'So muc: {len(muc)} | so anh rieng biet: {len(anh)}')

# Bat so gia tu mock: latency 0.0 nghia la khong chay model that
gia = [m for m in muc if float(m.get('latency_sec') or 0) == 0.0]
print(f'Muc co latency = 0 (dau hieu mock): {len(gia)}')
assert not gia, f'{len(gia)} muc chay mock -- so lieu khong dung duoc'

assert len(anh) >= 100, f'Chi {len(anh)} anh -- chua dat nguong de bai'
print('DAT nguong 100 anh cua de bai, khong co so gia.')


In [ ]:
import shutil

ZIP = shutil.make_archive('/kaggle/working/benchmark-355', 'zip', '/kaggle/working/ket_qua')
print('Da nen:', ZIP)
# Save & Run All tu luu version -- tai file zip o tab Output ve may.
